In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np
from pathlib import Path
import glob
import h5py

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.simulations.simulations import TNGSimulation

In [ ]:
base_path = "/home/dlopez/Documentos/0.profesional/Postdoc/USP/Projects/STORAGE/TNG50-4"
snapnum = 25

sim = TNGSimulation(base_path)

In [ ]:
# --------------------------------------------
# 1) High-level summary
# --------------------------------------------
print("\n=== Simulation summary ===")
print(sim.summary())

print("\n=== Snapshot numbers discovered ===")
print(sim.snapnums)

print("\n=== Common (stable) cosmology/metadata ===")
for k, v in sim.cosmo.items():
    if v is None:
        continue
    print(f"{k:24s}: {v}")

# --------------------------------------------
# 2) Inspect a particular snapshot
# --------------------------------------------
snapnum = sim.snapnums[len(sim.snapnums)//2]  # pick a middle snapshot, or set snapnum=25
info = sim.get(snapnum)

print(f"\n=== Snapshot {snapnum:03d} basic info ===")
print("snapdir:", info.snapdir)
print("nfiles :", len(info.files))
print("files  :")
for p in info.files[:10]:
    print("  ", p.name)
if len(info.files) > 10:
    print("  ...")

# --------------------------------------------
# 3) Cosmology dictionary for that snapshot
# --------------------------------------------
print(f"\n=== Snapshot {snapnum:03d} parsed cosmology ===")
for k, v in info.cosmo.items():
    if v is None:
        continue
    if isinstance(v, np.ndarray):
        v = v.tolist()
    print(f"{k:24s}: {v}")

# --------------------------------------------
# 4) Raw metadata keys (Header / Parameters / Config)
# --------------------------------------------
print(f"\n=== Snapshot {snapnum:03d} Header keys ===")
print(sorted(info.header.keys()))

print(f"\n=== Snapshot {snapnum:03d} Parameters keys ===")
print(sorted(info.parameters.keys()) if info.parameters else "(none)")

print(f"\n=== Snapshot {snapnum:03d} Config keys ===")
print(sorted(info.config.keys()) if info.config else "(none)")

# Optionally print specific raw header attrs commonly useful
want_header = [
    "BoxSize", "Redshift", "Time", "HubbleParam",
    "Omega0", "OmegaLambda", "OmegaBaryon",
    "NumFilesPerSnapshot", "NumPart_Total", "MassTable",
]
print(f"\n=== Snapshot {snapnum:03d} selected Header values (if present) ===")
for k in want_header:
    if k in info.header:
        v = info.header[k]
        if isinstance(v, np.ndarray):
            v = v.tolist()
        print(f"{k:24s}: {v}")

# --------------------------------------------
# 5) Loop over all snapshots: (snapnum, z, a) quick table
# --------------------------------------------
print("\n=== All snapshots: z and a ===")
print(f"{'snap':>6} {'z':>12} {'a':>12} {'nfiles':>8}")
for n in sim.snapnums:
    c = sim.get(n).cosmo
    z = c.get("Redshift", None)
    a = c.get("ScaleFactor", None)
    nf = len(sim.get(n).files)
    print(f"{n:6d} {str(z):>12} {str(a):>12} {nf:8d}")

In [ ]:
import numpy as np
import h5py

# ------------------------------------------------------------
# Checkpoint 2 (extended, sim-aware):
#   - assumes you already have: sim  (TNGSimulation instance)
#   - uses sim metadata instead of re-reading Header/Parameters
#   - reads only PartType0 datasets from the snapshot files
#   - returns everything in a dict: raw_fields_from_simulation_snap
# ------------------------------------------------------------

# ----------------------------
# User choices
# ----------------------------
snapnum = 25                 # set explicitly (or choose from sim.snapnums)
N_sample = 50_000            # how many gas cells to load from the *first* chunk
file_index = 0               # which chunk file to probe (0 is fine for structure/ranges)

# ----------------------------
# Snapshot info from sim
# ----------------------------
info = sim.get(snapnum)

# Metadata already parsed by sim
BoxSize = float(info.cosmo["BoxSize"])              # TNG convention: ckpc/h
z = float(info.cosmo["Redshift"])
a = float(info.cosmo["ScaleFactor"])
h = float(info.cosmo["h"])

# Unit scalars are already present in Header in your snapshot (your output shows they are there)
# In your sim class, these are stored under info.header / info.parameters.
# We *read from sim*, not from file:
UnitLength_in_cm = info.header.get("UnitLength_in_cm", None) or info.parameters.get("UnitLength_in_cm", None)
UnitMass_in_g = info.header.get("UnitMass_in_g", None) or info.parameters.get("UnitMass_in_g", None)
UnitVelocity_in_cm_per_s = info.header.get("UnitVelocity_in_cm_per_s", None) or info.parameters.get("UnitVelocity_in_cm_per_s", None)
UnitDensity_in_cgs = info.header.get("UnitDensity_in_cgs", None) or info.parameters.get("UnitDensity_in_cgs", None)

# Which file we actually open to load arrays
fn = info.files[file_index]
print("Using snapshot:", snapnum, " file:", fn.name)
print("z =", z, " a =", a, " h =", h, " BoxSize =", BoxSize)

# ----------------------------
# Helper: safe dataset read
# ----------------------------
def _read_slice(group, name, N):
    if name not in group:
        return None
    dset = group[name]
    # If dataset is smaller than N (unlikely), just read all
    nmax = min(N, dset.shape[0])
    return dset[:nmax]

# ----------------------------
# Load PartType0 fields
# ----------------------------
with h5py.File(fn, "r") as f:
    g = f["PartType0"]

    # Geometry / mesh
    coords = _read_slice(g, "Coordinates", N_sample)     # (N,3) ckpc/h
    dens   = _read_slice(g, "Density", N_sample)         # code density (comoving)
    mass   = _read_slice(g, "Masses", N_sample)          # code mass
    vol    = _read_slice(g, "Volume", N_sample)          # cell volume (often present for AREPO)
    if vol is None and (mass is not None) and (dens is not None):
        vol = mass / dens                                # fallback

    # Kinematics
    vel = _read_slice(g, "Velocities", N_sample)         # stored as km/s * sqrt(a) in TNG
    vpec = vel * np.sqrt(a) if vel is not None else None # peculiar km/s

    # Thermodynamics raw state (for temperature later)
    u  = _read_slice(g, "InternalEnergy", N_sample)      # code internal energy per unit mass
    xe = _read_slice(g, "ElectronAbundance", N_sample)   # electron abundance

    # Neutral hydrogen fraction (nHI = xHI * nH later)
    if "NeutralHydrogenAbundance" in g:
        xHI = _read_slice(g, "NeutralHydrogenAbundance", N_sample)
        xHI_name = "NeutralHydrogenAbundance"
    elif "NeutralHydrogenFraction" in g:
        xHI = _read_slice(g, "NeutralHydrogenFraction", N_sample)
        xHI_name = "NeutralHydrogenFraction"
    else:
        xHI = None
        xHI_name = None

    # Star formation activity (Rahmati-like workflows often use SFR>0 to flag SF gas)
    sfr = _read_slice(g, "StarFormationRate", N_sample)
    if sfr is None:
        sfr = np.zeros(coords.shape[0], dtype=np.float32) if coords is not None else None

    # Metal enrichment (for future Si abundance / ionization modeling)
    Z = _read_slice(g, "GFM_Metallicity", N_sample)      # scalar metallicity mass fraction
    metals = _read_slice(g, "GFM_Metals", N_sample)      # (N, Nelem) element mass fractions, if present

    # (Optional / sometimes helpful later)
    # - "SubfindDMDensity" etc. are not needed for line extraction.
    # - "SmoothingLength" may exist in some outputs (your Config has SAVE_HSML_IN_SNAPSHOT)
    hsml = _read_slice(g, "SmoothingLength", N_sample)

# ----------------------------
# Sanity checks + quick ranges
# ----------------------------
print("\n=== Quick sanity checks (sample) ===")
if coords is not None:
    print("coords:", coords.shape, coords.dtype)
    print("coords min:", coords.min(axis=0))
    print("coords max:", coords.max(axis=0))
    print("coords within [0, BoxSize):", np.all(coords >= 0) and np.all(coords < BoxSize))

if dens is not None:
    print("\ndens :", dens.shape, dens.dtype, " min/max:", float(dens.min()), float(dens.max()), " positive:", bool(np.all(dens > 0)))

if vol is not None:
    print("\nvol  :", vol.shape, vol.dtype, " min/max:", float(vol.min()), float(vol.max()), " positive:", bool(np.all(vol > 0)))

if xHI is not None:
    print(f"\nxHI  : {xHI.shape} {xHI.dtype} ({xHI_name}) min/max:", float(np.min(xHI)), float(np.max(xHI)))

if sfr is not None:
    print("\nsfr  :", sfr.shape, sfr.dtype, "  SFR>0 fraction:", float(np.mean(sfr > 0)))

if vel is not None:
    print("\nvel(stored):", vel.shape, vel.dtype)
    print("vpec(peculiar):", vpec.shape, " example first:", vpec[0])

if u is not None:
    print("\nu    :", u.shape, u.dtype, " min/max:", float(np.min(u)), float(np.max(u)))
if xe is not None:
    print("\nxe   :", xe.shape, xe.dtype, " min/max:", float(np.min(xe)), float(np.max(xe)))

if Z is not None:
    print("\nZ    :", Z.shape, Z.dtype, " min/max:", float(np.min(Z)), float(np.max(Z)))
if metals is not None:
    print("\nmetals:", metals.shape, metals.dtype, " (Nelem =", metals.shape[1], ")")

if hsml is not None:
    print("\nhsml :", hsml.shape, hsml.dtype, " min/median/max:", float(hsml.min()), float(np.median(hsml)), float(hsml.max()))

# Effective cell size (AREPO-like) if no hsml present
if (hsml is None) and (vol is not None):
    hsml_eff = np.cbrt(vol)
    print("\nEffective cell size hsml_eff ~ V^(1/3) (coords units):",
          "min/median/max =", float(hsml_eff.min()), float(np.median(hsml_eff)), float(hsml_eff.max()))
else:
    hsml_eff = None

# Convert coords to cMpc for convenience (ckpc/h -> cMpc = / (1000*h))
coords_cMpc = coords / (1000.0 * h) if coords is not None else None
L_box_cMpc = BoxSize / (1000.0 * h)
print("\nL_box [cMpc] =", L_box_cMpc)
if coords_cMpc is not None:
    print("coords_cMpc min:", coords_cMpc.min(axis=0))
    print("coords_cMpc max:", coords_cMpc.max(axis=0))

# ----------------------------
# Pack everything into a dict for later checkpoints
# ----------------------------
raw_fields_from_simulation_snap = {
    "meta": {
        "snapnum": snapnum,
        "file": str(fn),
        "BoxSize_ckpc_over_h": BoxSize,
        "L_box_cMpc": L_box_cMpc,
        "z": z,
        "a": a,
        "h": h,
        "UnitLength_in_cm": UnitLength_in_cm,
        "UnitMass_in_g": UnitMass_in_g,
        "UnitVelocity_in_cm_per_s": UnitVelocity_in_cm_per_s,
        "UnitDensity_in_cgs": UnitDensity_in_cgs,
    },
    "gas": {
        "Coordinates_ckpc_over_h": coords,
        "Coordinates_cMpc": coords_cMpc,
        "Density_code": dens,
        "Masses_code": mass,
        "Volume_code": vol,
        "SmoothingLength_code": hsml,         # if present in snapshot
        "SmoothingLength_eff_code": hsml_eff, # V^(1/3) fallback if hsml not present
        "Velocities_stored": vel,             # km/s * sqrt(a) in TNG
        "Velocities_peculiar_kms": vpec,      # km/s
        "InternalEnergy_code": u,
        "ElectronAbundance": xe,
        "NeutralHydrogen_field": xHI,
        "NeutralHydrogen_field_name": xHI_name,
        "StarFormationRate": sfr,
        "GFM_Metallicity": Z,
        "GFM_Metals": metals,
    }
}

print("\nraw_fields_from_simulation_snap keys:")
print(raw_fields_from_simulation_snap.keys())
print("gas keys:", raw_fields_from_simulation_snap["gas"].keys())
